In [148]:
from collections import defaultdict
from itertools import islice
from pathlib import Path
from random import sample
import csv

from drain3 import TemplateMiner
from drain3.masking import MaskingInstruction
from drain3.template_miner_config import TemplateMinerConfig
from random import Random


In [96]:
log_path = Path("../../data/loghub/HDFS_v1/HDFS.log")

with log_path.open(encoding="utf-8") as file:
    lines = list(islice(file, 1000))

Логи записаны в следуюшием формате:
дата время ID_потока уровень компонента текст_сообщения

In [97]:
class LogEntry:
    def __init__(self, line: str):
        self.raw_line = line.rstrip("\r\n")

        (
            self.date,
            self.time,
            self.pid,
            self.level,
            self.component,
            self.message,
        ) = self.raw_line.split(maxsplit=5)

In [98]:
entries = [LogEntry(line) for line in lines]

In [99]:
miner = TemplateMiner()
examples = defaultdict(list)

for entry in entries:
    result = miner.add_log_message(entry.message)
    examples[result["cluster_id"]].append(entry)

config file not found: drain3.ini


In [100]:
clusters = sorted(miner.drain.clusters, key=lambda cluster: cluster.size, reverse=True)

for cluster in clusters:
    print(f"ID: {cluster.cluster_id}, сообщений: {cluster.size}")
    print("Шаблон:", cluster.get_template())

    group_lines = examples[cluster.cluster_id]

    for entry in sample(group_lines, min(3, len(group_lines))):
        print("Пример:", entry.message)
    print()

ID: 1, сообщений: 455
Шаблон: Receiving block <*> src: <*> dest: <*>
Пример: Receiving block blk_-8013855621109800549 src: /10.251.43.115:52927 dest: /10.251.43.115:50010
Пример: Receiving block blk_-3940941529251864773 src: /10.251.197.226:42631 dest: /10.251.197.226:50010
Пример: Receiving block blk_-2372508938534145884 src: /10.250.19.227:34262 dest: /10.250.19.227:50010

ID: 10, сообщений: 306
Шаблон: <*> Served block <*> to <*>
Пример: 10.251.215.16:50010 Served block blk_-1608999687919862906 to /10.251.66.102
Пример: 10.251.39.179:50010 Served block blk_-3544583377289625738 to /10.250.18.114
Пример: 10.251.197.226:50010 Served block blk_-3544583377289625738 to /10.251.39.209

ID: 2, сообщений: 177
Шаблон: BLOCK* NameSystem.allocateBlock: <*> <*>
Пример: BLOCK* NameSystem.allocateBlock: /user/root/rand/_temporary/_task_200811092030_0001_m_000042_0/part-00042. blk_-40115644493265216
Пример: BLOCK* NameSystem.allocateBlock: /user/root/rand/_temporary/_task_200811092030_0001_m_000005

Обнаружили, что ":Transmitted" слито, из-за чего потерялось слово

In [101]:
class LogEntry:
    def __init__(self, line: str):
        self.raw_line = line.rstrip("\r\n")

        (
            self.date,
            self.time,
            self.pid,
            self.level,
            self.component,
            self.message,
        ) = self.raw_line.split(maxsplit=5)

        self.prepared_message = (
            self.message
            .replace(":Transmitted", ": Transmitted")
        )

In [102]:
entries = [LogEntry(line) for line in lines]

In [103]:
miner = TemplateMiner()
examples = defaultdict(list)

for entry in entries:
    result = miner.add_log_message(entry.prepared_message)
    examples[result["cluster_id"]].append(entry)

config file not found: drain3.ini


In [104]:
clusters = sorted(miner.drain.clusters, key=lambda cluster: cluster.size, reverse=True)

for cluster in clusters:
    print(f"ID: {cluster.cluster_id}, сообщений: {cluster.size}")
    print("Шаблон:", cluster.get_template())

    group_lines = examples[cluster.cluster_id]

    for entry in sample(group_lines, min(3, len(group_lines))):
        print("Пример:", entry.message)
    print()

ID: 1, сообщений: 455
Шаблон: Receiving block <*> src: <*> dest: <*>
Пример: Receiving block blk_-1871778753987286456 src: /10.250.11.100:58542 dest: /10.250.11.100:50010
Пример: Receiving block blk_5775293647706867248 src: /10.251.42.207:49929 dest: /10.251.42.207:50010
Пример: Receiving block blk_7352609810175565501 src: /10.251.194.213:36922 dest: /10.251.194.213:50010

ID: 10, сообщений: 306
Шаблон: <*> Served block <*> to <*>
Пример: 10.250.10.6:50010 Served block blk_-1608999687919862906 to /10.251.203.149
Пример: 10.251.215.16:50010 Served block blk_-1608999687919862906 to /10.251.66.102
Пример: 10.250.14.224:50010 Served block blk_-1608999687919862906 to /10.251.127.191

ID: 2, сообщений: 177
Шаблон: BLOCK* NameSystem.allocateBlock: <*> <*>
Пример: BLOCK* NameSystem.allocateBlock: /user/root/rand/_temporary/_task_200811092030_0001_m_000033_0/part-00033. blk_-7158210625015641227
Пример: BLOCK* NameSystem.allocateBlock: /user/root/rand/_temporary/_task_200811092030_0001_m_000072_

Теперь сделаем маскирование.

In [105]:
config = TemplateMinerConfig()

config.masking_instructions = [
    MaskingInstruction(r"\bblk_-?\d+\b", "BLOCK"),
    MaskingInstruction(r"\b(?:\d{1,3}\.){3}\d{1,3}(?::\d+)?\b", "ADDR"),
]

In [106]:
miner_masked = TemplateMiner(config=config)
examples_masked = defaultdict(list)

for entry in entries:
    result = miner_masked.add_log_message(entry.prepared_message)
    examples_masked[result["cluster_id"]].append(entry)

In [107]:
clusters_masked = sorted(miner_masked.drain.clusters, key=lambda cluster: cluster.size, reverse=True)

for cluster in clusters_masked:
    print(f"ID: {cluster.cluster_id}, сообщений: {cluster.size}")
    print("Шаблон:", cluster.get_template())

    group_lines = examples_masked[cluster.cluster_id]

    for entry in sample(group_lines, min(3, len(group_lines))):
        print("Пример:", entry.prepared_message)
    print()

ID: 1, сообщений: 455
Шаблон: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_2391384234620089834 src: /10.250.14.224:47170 dest: /10.250.14.224:50010
Пример: Receiving block blk_3098154771341398925 src: /10.251.193.224:54818 dest: /10.251.193.224:50010
Пример: Receiving block blk_7955481321933797799 src: /10.251.67.211:43682 dest: /10.251.67.211:50010

ID: 10, сообщений: 306
Шаблон: <ADDR> Served block <BLOCK> to /<ADDR>
Пример: 10.251.39.179:50010 Served block blk_-3544583377289625738 to /10.251.67.225
Пример: 10.250.14.224:50010 Served block blk_-1608999687919862906 to /10.251.75.143
Пример: 10.251.71.193:50010 Served block blk_-1608999687919862906 to /10.251.43.192

ID: 2, сообщений: 177
Шаблон: BLOCK* NameSystem.allocateBlock: <*> <BLOCK>
Пример: BLOCK* NameSystem.allocateBlock: /user/root/rand/_temporary/_task_200811092030_0001_m_000084_0/part-00084. blk_-50273257731426871
Пример: BLOCK* NameSystem.allocateBlock: /user/root/rand/_temporary/_task_200

Выделили через маски.

Но, теперь у нас слились Transmitted и Served. Скорректируем настройки

In [108]:
config = TemplateMinerConfig()

config.masking_instructions = [
    MaskingInstruction(r"\bblk_-?\d+\b", "BLOCK"),
    MaskingInstruction(r"\b(?:\d{1,3}\.){3}\d{1,3}(?::\d+)?\b", "ADDR"),
]

config.drain_sim_th = 0.6

In [109]:
miner_masked = TemplateMiner(config=config)
examples_masked = defaultdict(list)

for entry in entries:
    result = miner_masked.add_log_message(entry.prepared_message)
    examples_masked[result["cluster_id"]].append(entry)

In [113]:
clusters_masked = sorted(miner_masked.drain.clusters, key=lambda cluster: cluster.size, reverse=True)

for cluster in clusters_masked:
    print(f"ID: {cluster.cluster_id}, сообщений: {cluster.size}")
    print("Шаблон:", cluster.get_template())

    group_lines = examples_masked[cluster.cluster_id]

    for entry in sample(group_lines, min(3, len(group_lines))):
        print("Пример:", entry.prepared_message)
        print("Маска+:", miner_masked.masker.mask(entry.prepared_message))
    print()

ID: 1, сообщений: 455
Шаблон: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_-1608999687919862906 src: /10.250.14.224:35754 dest: /10.250.14.224:50010
Маска+: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_7063315473424667801 src: /10.251.127.191:50796 dest: /10.251.127.191:50010
Маска+: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_-1608999687919862906 src: /10.251.107.19:56341 dest: /10.251.107.19:50010
Маска+: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>

ID: 10, сообщений: 306
Шаблон: <ADDR> Served block <BLOCK> to /<ADDR>
Пример: 10.251.39.179:50010 Served block blk_-3544583377289625738 to /10.251.39.179
Маска+: <ADDR> Served block <BLOCK> to /<ADDR>
Пример: 10.251.197.226:50010 Served block blk_-3544583377289625738 to /10.251.75.79
Маска+: <ADDR> Served block <BLOCK> to /<ADDR>
Пример: 10.251.71.193:50010 Served block blk_-1608999687919862906 to /10.251.126.255
Маска+: <ADDR> Se

In [118]:
with log_path.open(encoding="utf-8") as file:
    lines = list(islice(file, 10000))


class LogEntry:
    def __init__(self, line: str):
        self.raw_line = line.rstrip("\r\n")

        (
            self.date,
            self.time,
            self.pid,
            self.level,
            self.component,
            self.message,
        ) = self.raw_line.split(maxsplit=5)

        self.prepared_message = (
            self.message
            .replace(":Transmitted", ": Transmitted")
        )


entries = [LogEntry(line) for line in lines]

config = TemplateMinerConfig()

config.masking_instructions = [
    # ID блока HDFS: blk_123 или blk_-123; знак минус необязателен.
    MaskingInstruction(r"\bblk_-?\d+\b", "BLOCK"),
    # IPv4-адрес с необязательным портом: 10.20.30.40 или 10.20.30.40:50010.
    MaskingInstruction(r"\b(?:\d{1,3}\.){3}\d{1,3}(?::\d+)?\b", "ADDR"),
]

config.drain_sim_th = 0.6

miner = TemplateMiner(config=config)
examples = defaultdict(list)

for entry in entries:
    result = miner.add_log_message(entry.prepared_message)
    examples[result["cluster_id"]].append(entry)

clusters = sorted(miner.drain.clusters, key=lambda cluster: cluster.size, reverse=True)

In [119]:
for cluster in clusters:
    print(f"ID: {cluster.cluster_id}, сообщений: {cluster.size}")
    print("Шаблон:", cluster.get_template())

    group_lines = examples[cluster.cluster_id]

    for entry in sample(group_lines, min(3, len(group_lines))):
        print("Пример:", entry.prepared_message)
        print("Маска+:", miner.masker.mask(entry.prepared_message))
    print()

ID: 1, сообщений: 3031
Шаблон: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_-1749022762966269981 src: /10.251.38.214:39993 dest: /10.251.38.214:50010
Маска+: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_2514531255881471188 src: /10.251.90.134:36084 dest: /10.251.90.134:50010
Маска+: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_-579646444174856708 src: /10.251.122.38:47875 dest: /10.251.122.38:50010
Маска+: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>

ID: 3, сообщений: 1851
Шаблон: PacketResponder <*> for block <BLOCK> terminating
Пример: PacketResponder 0 for block blk_6678472565712118087 terminating
Маска+: PacketResponder 0 for block <BLOCK> terminating
Пример: PacketResponder 2 for block blk_2825871331379974767 terminating
Маска+: PacketResponder 2 for block <BLOCK> terminating
Пример: PacketResponder 1 for block blk_-6704225939638932097 terminating
Маска+: PacketResponder 1 

На 10 000 сообщениях найдено 15 шаблонов.

Добавились 3 новых шаблона

In [121]:
with log_path.open(encoding="utf-8") as file:
    lines = list(islice(file, 100000))


class LogEntry:
    def __init__(self, line: str):
        self.raw_line = line.rstrip("\r\n")

        (
            self.date,
            self.time,
            self.pid,
            self.level,
            self.component,
            self.message,
        ) = self.raw_line.split(maxsplit=5)

        self.prepared_message = (
            self.message
            .replace(":Transmitted", ": Transmitted")
        )


entries = [LogEntry(line) for line in lines]

config = TemplateMinerConfig()

config.masking_instructions = [
    # ID блока HDFS: blk_123 или blk_-123; знак минус необязателен.
    MaskingInstruction(r"\bblk_-?\d+\b", "BLOCK"),
    # IPv4-адрес с необязательным портом: 10.20.30.40 или 10.20.30.40:50010.
    MaskingInstruction(r"\b(?:\d{1,3}\.){3}\d{1,3}(?::\d+)?\b", "ADDR"),
]

config.drain_sim_th = 0.6

miner = TemplateMiner(config=config)
examples = defaultdict(list)

for entry in entries:
    result = miner.add_log_message(entry.prepared_message)
    examples[result["cluster_id"]].append(entry)

clusters = sorted(miner.drain.clusters, key=lambda cluster: cluster.size, reverse=True)

In [123]:
for cluster in clusters:
    print(f"ID: {cluster.cluster_id}, сообщений: {cluster.size}")
    print("Шаблон:", cluster.get_template())

    group_lines = examples[cluster.cluster_id]

    for entry in sample(group_lines, min(2, len(group_lines))):
        print("Пример:", entry.prepared_message)
        print("Маска+:", miner.masker.mask(entry.prepared_message))
    print()

ID: 1, сообщений: 23611
Шаблон: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_-6487613344870651869 src: /10.251.111.130:52103 dest: /10.251.111.130:50010
Маска+: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_1913207732093518249 src: /10.251.70.112:39978 dest: /10.251.70.112:50010
Маска+: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>

ID: 5, сообщений: 22287
Шаблон: BLOCK* NameSystem.addStoredBlock: blockMap updated: <ADDR> is added to <BLOCK> size <*>
Пример: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.31.85:50010 is added to blk_8444270519748148509 size 67108864
Маска+: BLOCK* NameSystem.addStoredBlock: blockMap updated: <ADDR> is added to <BLOCK> size 67108864
Пример: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.70.37:50010 is added to blk_-1643361457253096128 size 67108864
Маска+: BLOCK* NameSystem.addStoredBlock: blockMap updated: <ADDR> is added to <BLOCK> size 67108864

ID: 4, сооб

In [126]:
for cluster in clusters:
    print("Шаблон:", cluster.get_template())

Шаблон: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Шаблон: BLOCK* NameSystem.addStoredBlock: blockMap updated: <ADDR> is added to <BLOCK> size <*>
Шаблон: Received block <BLOCK> of size <*> from /<ADDR>
Шаблон: PacketResponder <*> for block <BLOCK> terminating
Шаблон: BLOCK* NameSystem.allocateBlock: <*> <BLOCK>
Шаблон: Verification succeeded for <BLOCK>
Шаблон: <ADDR> Served block <BLOCK> to /<ADDR>
Шаблон: writeBlock <BLOCK> received exception java.io.IOException: Could not read from stream
Шаблон: Received block <BLOCK> src: /<ADDR> dest: /<ADDR> of size <*>
Шаблон: <ADDR>: Transmitted block <BLOCK> to /<ADDR>
Шаблон: Receiving empty packet for block <BLOCK>
Шаблон: BLOCK* ask <ADDR> to replicate <BLOCK> to datanode(s) <ADDR>
Шаблон: <ADDR> Starting thread to transfer block <BLOCK> to <ADDR>
Шаблон: Deleting block <BLOCK> file <*>
Шаблон: <ADDR> Starting thread to transfer block <BLOCK> to <ADDR>, <ADDR>
Шаблон: BLOCK* ask <ADDR> to replicate <BLOCK> to datanode(s) <ADDR> <A

Теперь обработаем всё

In [143]:
miner = TemplateMiner(config=config)
examples = defaultdict(list)
rng = Random(42)
processed_count = 0

In [144]:
with log_path.open(encoding="utf-8") as file:
    for line in file:
        entry = LogEntry(line)
        result = miner.add_log_message(entry.prepared_message)
        processed_count += 1

        group_examples = examples[result["cluster_id"]]

        if len(group_examples) < 3:
            group_examples.append(entry)
        else:
            position = rng.randrange(result["cluster_size"])

            if position < 3:
                group_examples[position] = entry

        if processed_count % 1_000_000 == 0:
            print(
                "Прочитано:", processed_count,
                "Шаблонов:", len(miner.drain.clusters),
                flush=True,
            )

Прочитано: 1000000 Шаблонов: 36
Прочитано: 2000000 Шаблонов: 39
Прочитано: 3000000 Шаблонов: 41
Прочитано: 4000000 Шаблонов: 41
Прочитано: 5000000 Шаблонов: 43
Прочитано: 6000000 Шаблонов: 43
Прочитано: 7000000 Шаблонов: 44
Прочитано: 8000000 Шаблонов: 45
Прочитано: 9000000 Шаблонов: 47
Прочитано: 10000000 Шаблонов: 47
Прочитано: 11000000 Шаблонов: 47


In [145]:
clusters = sorted(miner.drain.clusters, key=lambda cluster: cluster.size, reverse=True)

for cluster in clusters:
    print(f"ID: {cluster.cluster_id}, сообщений: {cluster.size}")
    print("Шаблон:", cluster.get_template())

    group_lines = examples[cluster.cluster_id]

    for entry in group_lines:
        print("Пример:", entry.prepared_message)
        print("Маска+:", miner.masker.mask(entry.prepared_message))
    print()

ID: 1, сообщений: 1723232
Шаблон: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_-5759642491490861565 src: /10.250.7.96:34834 dest: /10.250.7.96:50010
Маска+: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_1752372705221939149 src: /10.250.6.4:47546 dest: /10.250.6.4:50010
Маска+: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Пример: Receiving block blk_6259257855539924963 src: /10.251.107.19:50873 dest: /10.251.107.19:50010
Маска+: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>

ID: 5, сообщений: 1719741
Шаблон: BLOCK* NameSystem.addStoredBlock: blockMap updated: <ADDR> is added to <BLOCK> size <*>
Пример: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.250.7.96:50010 is added to blk_-6382121130895406799 size 67108864
Маска+: BLOCK* NameSystem.addStoredBlock: blockMap updated: <ADDR> is added to <BLOCK> size 67108864
Пример: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.42.9:50010 is added to b

In [157]:
for cluster in clusters:
    print("Шаблон:", cluster.get_template())

Шаблон: Receiving block <BLOCK> src: /<ADDR> dest: /<ADDR>
Шаблон: BLOCK* NameSystem.addStoredBlock: blockMap updated: <ADDR> is added to <BLOCK> size <*>
Шаблон: PacketResponder <*> for block <BLOCK> <*>
Шаблон: Received block <BLOCK> of size <*> from /<ADDR>
Шаблон: Deleting block <BLOCK> file <*>
Шаблон: BLOCK* NameSystem.delete: <BLOCK> is added to invalidSet of <ADDR>
Шаблон: BLOCK* NameSystem.allocateBlock: <*> <BLOCK>
Шаблон: <ADDR> Served block <BLOCK> to /<ADDR>
Шаблон: <ADDR>:Got exception while serving <BLOCK> to /<ADDR>:
Шаблон: Verification succeeded for <BLOCK>
Шаблон: Received block <BLOCK> src: /<ADDR> dest: /<ADDR> of size <*>
Шаблон: <ADDR>: Transmitted block <BLOCK> to /<ADDR>
Шаблон: BLOCK* ask <ADDR> to replicate <BLOCK> to datanode(s) <ADDR>
Шаблон: <ADDR> Starting thread to transfer block <BLOCK> to <ADDR>
Шаблон: Unexpected error trying to delete block <BLOCK>. BlockInfo not found in volumeMap.
Шаблон: writeBlock <BLOCK> received exception java.io.IOException: C

In [155]:
print("Прочитано:", processed_count)
print("Учтено:", sum(c.size for c in miner.drain.clusters))
print("Шаблонов:", len(miner.drain.clusters))

Прочитано: 11175629
Учтено: 11175629
Шаблонов: 47


Итого: 47 шаблонов

Есть <*> на которые стоит обратить внимание:
1. terminating / ...
2. java.io.EOFException / java.io.EOFException / ...

В каждой строке только один уникальный <BLOCK> (на строку)

Теперь вызываем Drain3.py